In [ ]:
pip install -U transformers


In [ ]:
pip install -U datasets trl peft accelerate bitsandbytes


In [ ]:
from datasets import load_dataset

In [ ]:
dataset = load_dataset(
    "HuggingFaceH4/ultrachat_200k",
    split="train_sft"
)

In [ ]:
print(dataset)

In [ ]:
# Shuffle and take 8000 random samples
train_dataset = dataset.shuffle(seed=42).select(range(4000))

In [ ]:
print(train_dataset)

In [ ]:
from transformers import AutoTokenizer

In [ ]:
model_name = "Qwen/Qwen3-4B"

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True,
)

In [ ]:
tokenizer.pad_token = tokenizer.eos_token
tokenizer.model_max_length = 512

In [ ]:
MAX_SEQ_LENGTH = 512

def format_chat(example):

    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )

    tokens = tokenizer(
        text,
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
    )

    text = tokenizer.decode(
        tokens["input_ids"],
        skip_special_tokens=False
    )

    return {
        "text": text
    }

In [ ]:
train_dataset = train_dataset.map(
    format_chat,
    remove_columns=dataset.column_names,
)


In [ ]:
print(train_dataset[0]["text"])

In [ ]:
from transformers import BitsAndBytesConfig
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

In [ ]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True,
)

In [ ]:
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)

model = prepare_model_for_kbit_training(model)

In [ ]:
model.gradient_checkpointing_enable()

model.config.use_cache = False

In [ ]:
peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",

    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)


In [ ]:
model = get_peft_model(
    model,
    peft_config,
)

In [ ]:
for name, param in model.named_parameters():
    if param.requires_grad:
        param.data = param.data.float()

In [ ]:
dtypes = {}

for name, param in model.named_parameters():
    if param.requires_grad:
        dtypes[str(param.dtype)] = dtypes.get(str(param.dtype), 0) + 1

print(dtypes)

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./qwen3_4b_sft",

    num_train_epochs=1,

    per_device_train_batch_size=1,

    gradient_accumulation_steps=16,

    learning_rate=2e-4,

    warmup_ratio=0.03,

    fp16=False,
    bf16=False,

    optim="paged_adamw_32bit",

    logging_steps=100,

    save_steps=500,

    save_total_limit=2,

    report_to="none",
)


In [ ]:
from trl import SFTTrainer

In [ ]:
trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    processing_class=tokenizer,
    args=training_args,
)

In [ ]:
print(
    f"Memory footprint: "
    f"{model.get_memory_footprint()/1024**3:.2f} GB"
)

In [ ]:
model.print_trainable_parameters()

In [ ]:
trainer.train()

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
import os

print(os.listdir("/content"))

In [ ]:
!unzip -q /content/qwen3_4b_sft.zip -d /content/model

In [ ]:
import os

for root, dirs, files in os.walk("/content/model"):
    print(root)
    print(files)

In [ ]:
trainer.save_model("./qwen3_4b_adapter")

tokenizer.save_pretrained(
    "./qwen3_4b_adapter"
)

In [ ]:
from datasets import load_dataset

dataset = load_dataset("robdixon/json-extraction")

print(dataset)
print(dataset["train"][0])


In [ ]:
columns_to_keep = [
    "instruction",
    "text",
    "json"
]

dataset["train"] = dataset["train"].remove_columns(
    [c for c in dataset["train"].column_names if c not in columns_to_keep]
)

dataset["validation"] = dataset["validation"].remove_columns(
    [c for c in dataset["validation"].column_names if c not in columns_to_keep]
)

In [ ]:
import json

def is_valid_json(example):
    try:
        json.loads(example["json"])
        return True
    except:
        return False

In [ ]:
train_dataset = dataset["train"].filter(is_valid_json)
val_dataset = dataset["validation"].filter(is_valid_json)

In [ ]:
train_dataset = train_dataset.shuffle(seed=42)
val_dataset = val_dataset.shuffle(seed=42)


In [ ]:
train_dataset = train_dataset.select(range(4000))
val_dataset = val_dataset.select(range(1000))

In [ ]:
print(val_dataset)

In [ ]:
def create_messages(example):

    user_message = (
        f"{example['instruction']}\n\n"
        f"Document:\n{example['text']}"
    )

    assistant_message = example["json"]

    return {
        "messages": [
            {
                "role": "user",
                "content": user_message
            },
            {
                "role": "assistant",
                "content": assistant_message
            }
        ]
    }

In [ ]:
train_dataset = train_dataset.map(
    create_messages,
    remove_columns=train_dataset.column_names
)

val_dataset = val_dataset.map(
    create_messages,
    remove_columns=val_dataset.column_names
)

In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM
)

base_model = "Qwen/Qwen3-4B"

tokenizer = AutoTokenizer.from_pretrained(base_model)

if tokenizer.pad_token is None:
  tokenizer.pad_token = tokenizer.eos_token



In [ ]:
import zipfile
import os
zip_path = "/content/qwen3_4b_sft.zip"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
  zip_ref.extractall()

In [ ]:
model_path = "/content/model/checkpoint-250"

In [ ]:
peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",

    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    base_model,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True,
)

In [ ]:
from peft import PeftModel

model = PeftModel.from_pretrained(model, model_path)

In [ ]:
model = model.merge_and_unload()

In [ ]:
instruction2_model = get_peft_model(model, peft_config)

In [ ]:
print(train_dataset[0])

In [ ]:
MAX_SEQ_LENGTH = 256

def formatting_func(example):

    user_text = example["messages"][0]["content"]
    assistant_text = example["messages"][1]["content"]

    text = (
        f"<|im_start|>user\n"
        f"{user_text}"
        f"<|im_end|>\n"
        f"<|im_start|>assistant\n"
        f"{assistant_text}"
        f"<|im_end|>"
    )

    tokenized = tokenizer(
        text,
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
    )

    text = tokenizer.decode(
        tokenized["input_ids"],
        skip_special_tokens=False
    )

    return {"text": text}

In [ ]:
train_dataset = train_dataset.map(
    formatting_func,
    remove_columns=["messages"]
)

val_dataset = val_dataset.map(
    formatting_func,
    remove_columns=["messages"]
)

In [ ]:
print(train_dataset[0]["text"])

In [ ]:
for name, param in instruction2_model.named_parameters():
    if param.requires_grad:
        param.data = param.data.float()

In [ ]:
dtypes = {}

for name, param in instruction2_model.named_parameters():
    if param.requires_grad:
        dtypes[str(param.dtype)] = dtypes.get(str(param.dtype), 0) + 1

print(dtypes)

In [ ]:
instruction2_model.gradient_checkpointing_enable()

instruction2_model.config.use_cache = False

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./qwen3_4b_sft_2",

    num_train_epochs=2,

    per_device_train_batch_size=1,

    gradient_accumulation_steps=16,

    learning_rate=2e-4,

    warmup_ratio=0.03,

    fp16=False,
    bf16=False,

    optim="paged_adamw_32bit",

    logging_steps=20,

    save_steps=500,

    save_total_limit=2,

    report_to="none",
)

In [ ]:
trainer = SFTTrainer(
    model=instruction2_model,
    train_dataset=train_dataset,
    processing_class=tokenizer,
    args=training_args,
)

In [ ]:
print(
    f"Memory footprint: "
    f"{model.get_memory_footprint()/1024**3:.2f} GB"
)

In [ ]:
trainer.train()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.DataFrame({
    "Step": [20,40,60,80,100,120,140,160,180,200,220,240],
    "Loss": [1.731622,1.182794,1.101851,1.056419,0.998329,
             1.037363,0.989463,0.997032,1.004074,
             0.978676,0.949651,0.988082]
})

plt.figure(figsize=(12,6))
plt.plot(df["Step"], df["Loss"], marker="o", linewidth=2)

plt.title("Supervised Fine-Tuning (SFT) Training Loss")
plt.xlabel("Training Step")
plt.ylabel("Loss")
plt.grid(alpha=0.2)

plt.tight_layout()
plt.show()

In [ ]:
import shutil

shutil.make_archive(
    "/content/qwen3_4b_sft_2",
    "zip",
    "/content/qwen3_4b_sft_2"
)

print("Zip file created!")

In [ ]:
from google.colab import files

files.download("/content/qwen3_4b_sft_2.zip")

In [ ]:
#Inference function
import torch

def generate_json(model, tokenizer, instruction, document):

    prompt = (
        f"{instruction}\n\n"
        f"Document:\n{document}"
    )

    messages = [
        {"role": "user", "content": prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(model.device)

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            temperature=0.0
        )

    prediction = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return prediction

In [ ]:
import re

def extract_json(text):

    match = re.search(
        r'\{.*\}',
        text,
        flags=re.DOTALL
    )

    if match:
        return match.group()

    return None

In [ ]:
#JSON Validity metric
import json

def is_valid_json(text):

    try:
        json.loads(text)
        return True
    except:
        return False

In [ ]:
#Exact match accuracy
def exact_match(pred_json, gt_json):

    return pred_json == gt_json

In [ ]:
#Field level accuracy
def field_accuracy(pred, gt):

    total = 0
    correct = 0

    for key in gt:

        total += 1

        if key in pred and pred[key] == gt[key]:
            correct += 1

    if total == 0:
        return 0

    return correct / total

In [ ]:
#Full Validation Loop
import json
from tqdm import tqdm
from datasets import Dataset

# ---------------------------
# Metrics
# ---------------------------

valid_json_count = 0
exact_match_count = 0

field_scores = []

# ---------------------------
# DPO Dataset Storage
# ---------------------------

dpo_pairs = []

NUM_SAMPLES = 500

for example in tqdm(val_dataset.select(range(NUM_SAMPLES))):

    # Generate prediction
    prediction_text = generate_json(
        instruction_sft_model,
        tokenizer,
        example["instruction"],
        example["text"]
    )

    # Extract JSON only
    json_text = extract_json(prediction_text)

    # Skip if JSON extraction failed
    if json_text is None:
        continue

    # Check JSON validity
    if is_valid_json(json_text):

        valid_json_count += 1

        pred_json = json.loads(json_text)

        gt_json = json.loads(example["json"])

        # ---------------------------
        # Normalized comparison
        # ---------------------------

        pred_norm = json.dumps(
            pred_json,
            sort_keys=True
        )

        gt_norm = json.dumps(
            gt_json,
            sort_keys=True
        )

        # ---------------------------
        # Exact Match
        # ---------------------------

        if pred_norm == gt_norm:

            exact_match_count += 1

        else:

            # =====================================
            # CREATE DPO PAIR ONLY FOR MISTAKES
            # =====================================

            prompt = (
                f"{example['instruction']}\n\n"
                f"Document:\n{example['text']}"
            )

            dpo_pairs.append(
                {
                    "prompt": prompt,
                    "chosen": example["json"],   # Ground Truth
                    "rejected": json_text        # Wrong SFT Prediction
                }
            )

        # ---------------------------
        # Field Accuracy
        # ---------------------------

        field_scores.append(
            field_accuracy(pred_json, gt_json)
        )



In [ ]:
# ===================================================
# CREATE DPO DATASET
# ===================================================

dpo_dataset = Dataset.from_list(dpo_pairs)

print("\nDPO Dataset Created")
print(dpo_dataset)

print("\nNumber of DPO Pairs:", len(dpo_dataset))

In [ ]:
# ===================================================
# FINAL METRICS
# ===================================================

json_validity_rate = valid_json_count / NUM_SAMPLES

exact_match_rate = exact_match_count / NUM_SAMPLES

avg_field_accuracy = (
    sum(field_scores) / len(field_scores)
    if len(field_scores) > 0
    else 0
)

print("=" * 60)
print(f"JSON Validity Rate : {json_validity_rate:.4f}")
print(f"Exact Match Rate   : {exact_match_rate:.4f}")
print(f"Field Accuracy     : {avg_field_accuracy:.4f}")
print("=" * 60)

In [ ]:
#Refusal correctness
refusal_examples = [
    "Hello how are you?",
    "What is machine learning?",
    "Tell me a joke."
]

In [ ]:
refusal_correct = 0

for text in refusal_examples:

    prediction = generate_json(
        model,
        tokenizer,
        "Extract JSON",
        text
    )

    if "cannot extract" in prediction.lower():
        refusal_correct += 1

refusal_accuracy = (
    refusal_correct / len(refusal_examples)
)

In [ ]:
results = pd.DataFrame({
    "Model": ["Base Qwen", "SFT"],
    "JSON Validity": [0.00, 0.9880],
    "Exact Match": [0.00, 0.0480],
    "Field Accuracy": [0.00, 0.1648]
})

results

In [ ]:
#DPO

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
import os

print(os.listdir("/content"))

In [ ]:
!unzip -q /content/qwen3_4b_dpo -d /content/

In [ ]:
import os

for root, dirs, files in os.walk("/content/model_2"):
    print(root)
    print(files)

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
import pandas as pd

df = pd.read_csv("json_dpo_dataset.csv")

df = df.drop_duplicates(
    subset=["prompt", "chosen", "rejected"]
)

print(len(df))

In [ ]:
df = df[
    (df["chosen"].str.len() > 2)
    &
    (df["rejected"].str.len() > 2)
]

In [ ]:
from datasets import Dataset

dpo_dataset = Dataset.from_pandas(
    df,
    preserve_index=False
)

print(dpo_dataset)

In [ ]:
dpo_split = dpo_dataset.train_test_split(
    test_size=0.1,
    seed=42
)

dpo_train = dpo_split["train"]
dpo_eval = dpo_split["test"]

print(len(dpo_train))
print(len(dpo_eval))

In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM
)

base_model = "Qwen/Qwen3-4B"

tokenizer = AutoTokenizer.from_pretrained(base_model)

if tokenizer.pad_token is None:
  tokenizer.pad_token = tokenizer.eos_token

In [ ]:
from transformers import BitsAndBytesConfig
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

In [ ]:
import zipfile
import os
zip_path = "/content/qwen3_4b_sft_2.zip"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
  zip_ref.extractall()

In [ ]:
model_path = "/content/model/checkpoint-250"

In [ ]:
from peft import LoraConfig

peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",

    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    base_model,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True,
)

In [ ]:
from peft import (
    PeftModel,
    get_peft_model
)

model = PeftModel.from_pretrained(model, model_path)

In [ ]:
model = model.merge_and_unload()

In [ ]:
dpo_model = get_peft_model(model, peft_config)

In [ ]:
for name, param in dpo_model.named_parameters():
    if param.requires_grad:
        param.data = param.data.float()

In [ ]:
dtypes = {}

for name, param in dpo_model.named_parameters():
    if param.requires_grad:
        dtypes[str(param.dtype)] = dtypes.get(str(param.dtype), 0) + 1

print(dtypes)

In [ ]:
from trl import DPOConfig

training_args = DPOConfig(
    output_dir="qwen3_4b_dpo",

    num_train_epochs=2,

    per_device_train_batch_size=1,

    gradient_accumulation_steps=16,

    learning_rate=5e-6,

    lr_scheduler_type="cosine",

    warmup_ratio=0.1,

    logging_steps=10,

    save_steps=10,

    eval_steps=10,

    eval_strategy="steps",

    save_strategy="steps",

    bf16=False,

    fp16=False,

    beta=0.1,

    max_length=256,

    remove_unused_columns=False,

    report_to="none",

    gradient_checkpointing=True
)

In [ ]:
dpo_model.gradient_checkpointing_enable()

In [ ]:
print(next(dpo_model.parameters()).dtype)

In [ ]:
for name, param in dpo_model.named_parameters():
    print(name, param.dtype)
    break

In [ ]:
dpo_model.config.use_cache = False

In [ ]:
from trl import DPOTrainer

In [ ]:
trainer = DPOTrainer(
    model=dpo_model,
    args=training_args,
    ref_model=None,
    train_dataset=dpo_train,
    eval_dataset=dpo_eval,
    processing_class=tokenizer
)

In [ ]:
print(
    f"Memory footprint: "
    f"{model.get_memory_footprint()/1024**3:.2f} GB"
)

In [ ]:
trainer.train()

In [ ]:
import matplotlib.pyplot as plt

# Data from DPO training table
steps = [10, 20, 30, 40, 50, 54]

training_loss = [
    0.692859,
    0.688443,
    0.679924,
    0.672331,
    0.668146,
    0.668146
]

plt.figure(figsize=(10, 6))

plt.plot(
    steps,
    training_loss,
    marker='o',
    linewidth=2
)

plt.title("Qwen3-4B DPO Training Loss Curve")
plt.xlabel("Training Step")
plt.ylabel("Training Loss")
plt.grid(True)

# Show exact values on points
for x, y in zip(steps, training_loss):
    plt.annotate(
        f"{y:.4f}",
        (x, y),
        textcoords="offset points",
        xytext=(0, 8),
        ha='center'
    )

plt.show()

In [ ]:
import shutil

shutil.make_archive(
    "/content/qwen3_4b_dpo",
    "zip",
    "/content/qwen3_4b_dpo"
)

print("Zip file created!")

In [ ]:
from google.colab import files

files.download("/content/qwen3_4b_dpo.zip")

In [ ]:
path = "/content/checkpoint-54"

In [ ]:
!pip uninstall -y torchao

In [ ]:
final_dpo_model = AutoModelForCausalLM.from_pretrained(path, device_map = "cuda")

In [ ]:
def generate_json(model, tokenizer, instruction, text):

    prompt = (
        f"{instruction}\n\n"
        f"Document:\n{text}"
    )

    messages = [
        {"role": "user", "content": prompt}
    ]

    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        formatted_prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=False,
        temperature=0.0
    )

    response = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return response

In [ ]:
import re
import json

def extract_json(text):

    match = re.search(
        r"\{.*\}",
        text,
        flags=re.DOTALL
    )

    if match:
        return match.group()

    return None

In [ ]:
def is_valid_json(text):

    try:
        json.loads(text)
        return True

    except:
        return False

In [ ]:
def exact_match(pred_json, gt_json):

    pred_norm = json.dumps(
        pred_json,
        sort_keys=True
    )

    gt_norm = json.dumps(
        gt_json,
        sort_keys=True
    )

    return pred_norm == gt_norm

In [ ]:
def field_accuracy(pred_json, gt_json):

    total_fields = len(gt_json)

    if total_fields == 0:
        return 0

    correct_fields = 0

    for key, value in gt_json.items():

        if key in pred_json:

            if pred_json[key] == value:
                correct_fields += 1

    return correct_fields / total_fields

In [ ]:
import json
from tqdm import tqdm

valid_json_count = 0
exact_match_count = 0

field_scores = []

NUM_SAMPLES = 100

for example in tqdm(val_dataset.select(range(NUM_SAMPLES))):

    prediction_text = generate_json(
        final_dpo_model,
        tokenizer,
        example["instruction"],
        example["text"]
    )

    json_text = extract_json(prediction_text)

    if json_text is None:
        continue

    if is_valid_json(json_text):

        valid_json_count += 1

        pred_json = json.loads(json_text)

        gt_json = json.loads(example["json"])

        if exact_match(pred_json, gt_json):
            exact_match_count += 1

        field_scores.append(
            field_accuracy(
                pred_json,
                gt_json
            )
        )

In [ ]:
json_validity_rate = (
    valid_json_count / NUM_SAMPLES
)

exact_match_rate = (
    exact_match_count / NUM_SAMPLES
)

avg_field_accuracy = (
    sum(field_scores) / len(field_scores)
    if len(field_scores) > 0
    else 0
)

print("=" * 60)
print("DPO EVALUATION")
print("=" * 60)

print(
    f"JSON Validity Rate : {json_validity_rate:.4f}"
)

print(
    f"Exact Match Rate   : {exact_match_rate:.4f}"
)

print(
    f"Field Accuracy     : {avg_field_accuracy:.4f}"
)

print("=" * 60)

In [ ]:
results = pd.DataFrame(
    {
        "Metric": [
            "JSON Validity",
            "Exact Match",
            "Field Accuracy"
        ],

        "SFT": [
            0.9880,
            0.0480,
            0.1648
        ],

        "DPO": [
            json_validity_rate,
            exact_match_rate,
            avg_field_accuracy
        ]
    }
)

print(results)

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
results.set_index("Metric").plot(
    kind="bar",
    figsize=(8,5)
)

plt.ylabel("Score")
plt.title("SFT vs DPO Performance")
plt.xticks(rotation=0)

plt.show()

In [ ]:
import matplotlib.pyplot as plt

# SFT
sft_steps = [20,40,60,80,100,120,140,160,180,200,220,240]
sft_loss = [
    1.731622,
    1.182794,
    1.101851,
    1.056419,
    0.998329,
    1.037363,
    0.989463,
    0.997032,
    1.004074,
    0.978676,
    0.949651,
    0.988082
]

# DPO
dpo_steps = [10,20,30,40,50,54]
dpo_loss = [
    0.692859,
    0.688443,
    0.679924,
    0.672331,
    0.668146,
    0.668146
]

plt.figure(figsize=(12,6))

plt.plot(
    sft_steps,
    sft_loss,
    marker='o',
    label='SFT Training Loss'
)

plt.plot(
    dpo_steps,
    dpo_loss,
    marker='s',
    label='DPO Training Loss'
)

plt.title("Qwen3-4B Fine-Tuning Progress")
plt.xlabel("Training Step")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)

plt.show()